# Figure SS02 (supplementary): the 3Di-entropy stripes in smORFs are two- and three-state strings

**Status:** draft, Gadi. The ID is provisional.

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figSS02_stripe_bands.tsv.gz` | `stripes/stripe_bands.tsv` | `stripe_states.py` (job 181047298) |

- **Run:** `linsalrob/genome_entropy` `Validation/smorfs/` (#105), with the write-up at `agent-handoffs` `projects/genome_entropy_logan/smorfs/manuscript.md`.
  - Software: `genome_entropy` 0.3.0, the ModernProst `gbouras13/modernprost-50M` encoder at revision `2d68d506`, encoding size 50,000, and translation table 11.
  - Hardware: V100 (`gpuvolta`).
- **Results location:** Gadi `/g/data/ob80/re3494/Projects/smORFs/results/`, mirrored on Teams at `genome_entropy:General/smORFs/results/`.
- **Reduction:** `manuscript/prep/smorfs_prepare_data.py`. It produces histograms over the per-smORF table (PBS job 181367025, 0.06 SU) and copies the AUC, model and stripe tables verbatim.
- **Data:** the GMSC benchmark sets as supplied by the GMSC authors.
  - **Positives:** 33,647,794 high-quality smORFs with RNAcode plus transcriptomic or proteomic support, or homology to known small proteins.
  - **Negatives:** 10,079,115 conserved, in-frame GMSC sequences that failed coding tests.
  - Lengths are 10–99 aa, with no 90 aa floor. The negatives are not random ORFs.
- **Bands:** strings within ±0.0055 bits of log₂k (k = 2, 3, 7), with control bands of the same width centred 0.03 bits below and above.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}

import math
from matplotlib.colors import LogNorm, TwoSlopeNorm
CLS = {"positive": "#0072B2", "negative": "#D55E00"}


In [ ]:
S = pd.read_csv(DATA / "figSS02_stripe_bands.tsv.gz", sep="\t")
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4), sharey=False, gridspec_kw={"wspace": 0.35})
BANDS = ["below", "stripe", "above"]
w = 0.38
for ax, k, letter in zip(axes, (2, 3, 7), "abc"):
    for i, c in enumerate(("positive", "negative")):
        g = S[(S.dataset == c) & (S.k == k)].set_index("band").reindex(BANDS)
        y = g.pct_with_k_states if k != 7 else g.per_million
        xs = np.arange(3) + (i - 0.5) * w
        ax.bar(xs, y, width=w, color=CLS[c], label=c)
        for xx, v in zip(xs, y):
            ax.text(xx, v, f"{v:.0f}" if k == 7 else f"{v:.0f}%", ha="center", va="bottom",
                    fontsize=5.6, color=INK)
    c0 = S[(S.k == k) & (S.band == "stripe")].centre_bits.iloc[0]
    ax.set_xticks(range(3), ["−0.03 bits", f"stripe\n{c0:.3f}", "+0.03 bits"])
    if k == 7:
        ax.set_ylabel("strings per million (band)")
        ax.set_title("k = 7: no stripe at log$_2$7", loc="left", fontsize=7)
    else:
        ax.set_ylabel(f"% of strings using exactly {k} states")
        ax.set_title(f"k = {k}: stripe at log$_2${k}", loc="left", fontsize=7)
    panel(ax, letter)
axes[0].legend(frameon=False, fontsize=6)
for ax in axes[:2]:
    ax.set_ylim(0, 80)
save(fig, "figSS02_smorf_stripes")


In [ ]:
print(S[["dataset", "k", "band", "n_band", "per_million", "pct_with_k_states", "top_state_set", "pct_top_state_set"]].to_string(index=False))


## Caption (draft)

**Figure SS02. The sharp 3Di-entropy stripes in smORFs come from the log₂k ceiling on strings that use only k states. They are not a biological threshold.**

**(a)** At the 1.0-bit stripe, 67% of positive and 61% of negative strings use exactly two 3Di states, almost always D and P. The control band 0.03 bits above contains none.

**(b)** At the 1.585-bit stripe, 42% of positive and 31% of negative strings use exactly three states, mostly D, P and V.

**(c)** At log₂7 = 2.807 bits the density is the same as in the flanking bands (2,388 against 2,446 and 2,406 per million positives), and every string there uses 11–15 states.

Both classes contain few-state strings at similar rates, so low 3Di entropy alone does not mark a sequence as non-coding. Which structural states D, P and V correspond to has not been measured here; see the decision register.
